# 10 — Phase A envelope scan (PB-0 two-stage flow)

**Purpose**: surface the envelope of what is observable from the multi-TPV mission within the configurable flight-time budget and the satellite-coincidence hard constraints (`T_dep >= 0`, `arc_km >= SAT_MIN_LENGTH_KM`).

**Owner**: FPO-6 / PB-0 (FPO engineer).

**Two-stage flow (PB-0 change)** — per Master 2026-06-09 "model run and rendering should be separable":

* **Stage 1 — Run model + snapshot** (cell 1, ~4 min) — load real data, run `phase_a_envelope`, save the result to `archive.pkl` via `save_phase_a_run`.
* **Stage 2 — Render figures from snapshot** (cell 2, ~5 s) — `load_phase_a_run`, then call `render_envelope` helpers. Re-run this cell to iterate on figure style without paying the 4 minute model cost.

The mission profile sits at the top of cell 1 (777 / 12 h defaults; G3 / 5 h reproducible by editing the same parameters back).

**LOCK 1**: every plan surfaced satisfies the budget, routes around every restricted polygon, and admits at least one valid satellite arc.

**S2-min scope**: satellite enters as a hard constraint only — coinc_dist optimisation belongs to Phase B.

**D5 verifier**: every plan in B1/B2 is independently re-checked against the raw shapefiles before display.

In [ ]:
%matplotlib inline
from shapely.ops import unary_union

# =====================================================================
# Mission profile - edit BEFORE each run.
# Default = Boeing 777 cruise, 12 h sortie (2026 mission set).
# G3 / 5 h baseline (07's reference): SPEED=850, HOURS=5, TURN_TIME=7.5
# =====================================================================
AIRCRAFT_SPEED_KMH = 905.0   # 777 cruise (M0.84). G3: 850
TOTAL_FLIGHT_HOURS = 12.0    # new mission default. G3: 5.0
TURN_TIME_MIN      = 7.5     # G3 idiom kept until 777 field measurement says otherwise
SAT_MIN_TIME_MIN   = 6.0     # spec; ground track coverage floor
SAT_IDEAL_TIME_MIN = 30.0    # spec; ground track coverage ideal

BUDGET_KM         = AIRCRAFT_SPEED_KMH * TOTAL_FLIGHT_HOURS               # auto
TURN_PENALTY_KM   = TURN_TIME_MIN / 60.0 * AIRCRAFT_SPEED_KMH             # auto
SAT_MIN_LENGTH_KM = SAT_MIN_TIME_MIN / 60.0 * AIRCRAFT_SPEED_KMH          # auto
SAT_ARC_MAX_KM    = SAT_IDEAL_TIME_MIN / 60.0 * AIRCRAFT_SPEED_KMH        # auto

ARCHIVE_PATH = 'phase_a_archive.pkl'   # PB-0 snapshot destination

print('Mission profile')
print(f'  AIRCRAFT_SPEED_KMH  = {AIRCRAFT_SPEED_KMH:.0f}')
print(f'  TOTAL_FLIGHT_HOURS  = {TOTAL_FLIGHT_HOURS:.1f}')
print(f'  TURN_TIME_MIN       = {TURN_TIME_MIN:.1f}')
print(f'  -> BUDGET_KM         = {BUDGET_KM:.0f}')
print(f'  -> TURN_PENALTY_KM   = {TURN_PENALTY_KM:.1f} (per sharp turn)')
print(f'  -> SAT_MIN_LENGTH_KM = {SAT_MIN_LENGTH_KM:.1f}')
print(f'  -> SAT_ARC_MAX_KM    = {SAT_ARC_MAX_KM:.1f}')

from multi_target_planner import (
    DEFAULT_DATA_DIR, TPV_FILENAME_PHASE0A,
    verify_data_sources,
    load_tpvs, load_restricted, load_atc, load_dropsonde,
    load_satellite_track, load_gatepoints, base_km,
    phase_a_envelope, format_envelope_summary,
    build_run, save_phase_a_run,
)

# ----- Stage 1a: real-data load -----
report = verify_data_sources(DEFAULT_DATA_DIR)
missing = [(r, p) for r, (p, ok) in report.items() if not ok]
if missing:
    raise FileNotFoundError(f'Missing real-data inputs: {missing}')

tpvs, frame = load_tpvs(DEFAULT_DATA_DIR, TPV_FILENAME_PHASE0A)
restricted = load_restricted(DEFAULT_DATA_DIR, frame)
atc        = load_atc(DEFAULT_DATA_DIR, frame)
dropsonde  = load_dropsonde(DEFAULT_DATA_DIR, frame)
satellite  = load_satellite_track(DEFAULT_DATA_DIR, frame)
gatepoints = load_gatepoints(DEFAULT_DATA_DIR, frame)
base       = base_km(frame)
restricted_union = unary_union(restricted) if restricted else None
atc_union        = unary_union(atc) if atc else None
print(f'\nTPVs: {len(tpvs)}   gatepoints: {gatepoints.shape[0]}   restricted polys: {len(restricted)}')

# ----- Stage 1b: run the model (~4 min on a cold run) -----
result = phase_a_envelope(
    tpvs=tpvs,
    base_km=base,
    gatepoints_km=gatepoints,
    restricted_union=restricted_union,
    atc_union=atc_union,
    sat_track_xy=satellite,
    enforce_sat_constraint=True,
    n_chord_options=(1, 2, 3, 4),
    angle_devs_deg=(-5.0, 0.0, 5.0),
    min_spacing_km=100.0,
    budget_km=BUDGET_KM,
    turn_penalty_km=TURN_PENALTY_KM,
    aircraft_speed_kmh=AIRCRAFT_SPEED_KMH,
    sat_min_length_km=SAT_MIN_LENGTH_KM,
    sat_arc_max_km=SAT_ARC_MAX_KM,
    k_per_bundle=999,   # let the bundles surface the entire archive; rendering decides which to show
)
print(format_envelope_summary(result, [t.label for t in tpvs]))

# ----- Stage 1c: D5 verifier on every B1+B2 plan -----
from multi_target_planner import IndependentMissionState, verify_bundle
state = IndependentMissionState.from_disk()
for name, bundle in [('B1', result.bundle_b1), ('B2', result.bundle_b2)]:
    reports = verify_bundle(bundle.plans, state=state,
                            budget_km=BUDGET_KM,
                            turn_penalty_km=TURN_PENALTY_KM,
                            aircraft_speed_kmh=AIRCRAFT_SPEED_KMH,
                            sat_min_length_km=SAT_MIN_LENGTH_KM)
    for i, (p, r) in enumerate(zip(bundle.plans, reports), 1):
        tag = 'OK ' if r.ok else 'FAIL'
        if not r.ok:
            print(f'  {name}#{i}: {tag} independent_cost={r.independent_cost_km:.0f} km violations={list(r.violations)}')

# ----- Stage 1d: snapshot to disk so rendering does not need a rerun -----
run = build_run(
    envelope=result,
    tpvs=tpvs, frame=frame, base_km=base, gatepoints_km=gatepoints,
    restricted_polygons=restricted, atc_polygons=atc, dropsonde_polygons=dropsonde,
    sat_track_xy=satellite,
    mission_profile={
        'AIRCRAFT_SPEED_KMH': AIRCRAFT_SPEED_KMH,
        'TOTAL_FLIGHT_HOURS': TOTAL_FLIGHT_HOURS,
        'TURN_TIME_MIN':      TURN_TIME_MIN,
        'BUDGET_KM':          BUDGET_KM,
        'TURN_PENALTY_KM':    TURN_PENALTY_KM,
        'SAT_MIN_LENGTH_KM':  SAT_MIN_LENGTH_KM,
        'SAT_ARC_MAX_KM':     SAT_ARC_MAX_KM,
    },
    notes={'tpv_filename': TPV_FILENAME_PHASE0A},
)
digest = save_phase_a_run(run, ARCHIVE_PATH)
print(f'\nSnapshot: {ARCHIVE_PATH}  (sha1 {digest[:12]}...)')

In [ ]:
# Stage 2: load snapshot and render figures.
# Re-run THIS cell only to iterate on the figure style without paying the 4-min model cost.
from multi_target_planner import load_phase_a_run
import render_envelope as renderer
import importlib
importlib.reload(renderer)   # picks up edits to render_envelope.py in dev mode

ARCHIVE_PATH = ARCHIVE_PATH if 'ARCHIVE_PATH' in dir() else 'phase_a_archive.pkl'
run = load_phase_a_run(ARCHIVE_PATH)

# Master's γ pick: one grid view + per-plan detail figures.
grid_path = renderer.render_grid(run, '10_phase_a_envelope.png')
print('Saved', grid_path)

# Single-plan detail for plan #1 (cheapest single-TPV plan).
detail_path = renderer.render_plan_detail(run, plan_idx=0, out_path='10_phase_a_plan_01_detail.png')
print('Saved', detail_path)